In [1]:
import mlflow
from mlflow import MlflowClient

MLFLOW_TRACKING_URI = "http://mlflow:5000"
MODEL_NAME = "finpulse-churn-catboost"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

versions = client.search_model_versions(
    filter_string=f"name='{MODEL_NAME}'"
)

print(f"Modelo registrado: {MODEL_NAME}")
print(f"Quantidade de versões: {len(versions)}")

for model_version in sorted(versions, key=lambda item: int(item.version)):
    print(
        f"Versão: {model_version.version} | "
        f"Run ID: {model_version.run_id} | "
        f"Status: {model_version.status}"
    )

Modelo registrado: finpulse-churn-catboost
Quantidade de versões: 3
Versão: 1 | Run ID:  | Status: READY
Versão: 2 | Run ID:  | Status: READY
Versão: 3 | Run ID: aebd3368a75b4f9ca629132e11234b66 | Status: READY


In [2]:
CANDIDATE_VERSION = "3"

model_version = client.get_model_version(
    name=MODEL_NAME,
    version=CANDIDATE_VERSION
)

print("=== VERSÃO CANDIDATA ===")
print(f"Modelo: {model_version.name}")
print(f"Versão: {model_version.version}")
print(f"Status: {model_version.status}")
print(f"Run ID: {model_version.run_id}")
print(f"Source: {model_version.source}")
print(f"Aliases atuais: {model_version.aliases}")

run = client.get_run(model_version.run_id)

print("\n=== RUN DE ORIGEM ===")
print(f"Run name: {run.data.tags.get('mlflow.runName')}")
print(f"Status: {run.info.status}")

print("\n=== MÉTRICAS ===")
for metric_name, metric_value in sorted(run.data.metrics.items()):
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== ARTEFATOS NA RAIZ ===")
for artifact in client.list_artifacts(model_version.run_id):
    artifact_type = "diretório" if artifact.is_dir else "arquivo"
    print(f"{artifact.path} — {artifact_type}")

=== VERSÃO CANDIDATA ===
Modelo: finpulse-churn-catboost
Versão: 3
Status: READY
Run ID: aebd3368a75b4f9ca629132e11234b66
Source: models:/m-42da499887dd46b19479736991e5cfe8
Aliases atuais: ['champion']

=== RUN DE ORIGEM ===
Run name: catboost-final-test
Status: FINISHED

=== MÉTRICAS ===
test_accuracy: 0.975321
test_average_precision: 0.969930
test_balanced_accuracy: 0.941745
test_f1: 0.920635
test_precision: 0.950820
test_recall: 0.892308
test_roc_auc: 0.993370
test_watchlist_accuracy: 0.963475
test_watchlist_average_precision: 0.969930
test_watchlist_balanced_accuracy: 0.955847
test_watchlist_f1: 0.892442
test_watchlist_precision: 0.845730
test_watchlist_recall: 0.944615
test_watchlist_roc_auc: 0.993370

=== ARTEFATOS NA RAIZ ===
evaluation — diretório
metadata — diretório


In [3]:
MODEL_URI = f"models:/{MODEL_NAME}@champion"

champion_version = client.get_model_version_by_alias(
    name=MODEL_NAME,
    alias="champion"
)

print("=== RESOLUÇÃO DO ALIAS ===")
print(f"URI estável: {MODEL_URI}")
print(f"Versão resolvida: {champion_version.version}")
print(f"Run ID: {champion_version.run_id}")
print(f"Source: {champion_version.source}")

print("\n=== LOAD-BACK PELO MODEL REGISTRY ===")

loaded_model = mlflow.pyfunc.load_model(MODEL_URI)

print("Modelo recuperado com sucesso!")
print(f"Tipo carregado: {type(loaded_model)}")
print(f"Model UUID: {loaded_model.metadata.model_uuid}")
print(f"Run ID: {loaded_model.metadata.run_id}")
print(f"Signature: {loaded_model.metadata.signature}")

=== RESOLUÇÃO DO ALIAS ===
URI estável: models:/finpulse-churn-catboost@champion
Versão resolvida: 3
Run ID: aebd3368a75b4f9ca629132e11234b66
Source: models:/m-42da499887dd46b19479736991e5cfe8

=== LOAD-BACK PELO MODEL REGISTRY ===


Modelo recuperado com sucesso!
Tipo carregado: <class 'mlflow.pyfunc.PyFuncModel'>
Model UUID: m-42da499887dd46b19479736991e5cfe8
Run ID: aebd3368a75b4f9ca629132e11234b66
Signature: inputs: 
  ['customer_age': long (required), 'gender': string (required), 'dependent_count': long (required), 'education_level': string (required), 'marital_status': string (required), 'income_category': string (required), 'card_category': string (required), 'months_on_book': long (required), 'total_relationship_count': long (required), 'months_inactive_last_12m': long (required), 'contacts_count_last_12m': long (required), 'credit_limit': double (required), 'total_revolving_balance': double (required), 'average_open_to_buy': double (required), 'amount_change_q4_q1': double (required), 'total_transaction_amount': double (required), 'total_transaction_count': long (required), 'transaction_count_change_q4_q1': double (required), 'average_utilization_ratio': double (required)]
outputs: 
  [Tensor('int64', (-1,

In [4]:
import os
import numpy as np
import pandas as pd

from sqlalchemy import create_engine

DATABASE_URL = os.getenv(
    "FINPULSE_DATABASE_URL",
    (
        "postgresql+psycopg2://"
        "finpulse_user:finpulse_password"
        "@postgres:5432/finpulse"
    ),
)

engine = create_engine(DATABASE_URL)

FEATURE_COLUMNS = [
    "customer_age",
    "gender",
    "dependent_count",
    "education_level",
    "marital_status",
    "income_category",
    "card_category",
    "months_on_book",
    "total_relationship_count",
    "months_inactive_last_12m",
    "contacts_count_last_12m",
    "credit_limit",
    "total_revolving_balance",
    "average_open_to_buy",
    "amount_change_q4_q1",
    "total_transaction_amount",
    "total_transaction_count",
    "transaction_count_change_q4_q1",
    "average_utilization_ratio",
]

query = """
SELECT *
FROM marts.mart_customer_churn_model
ORDER BY customer_id
LIMIT 10
"""

sample_data = pd.read_sql(query, engine)
X_sample = sample_data[FEATURE_COLUMNS].copy()

# PyFunc: valida o contrato padrão do MLflow.
pyfunc_predictions = loaded_model.predict(X_sample)

# Flavor nativo: permite acessar predict_proba.
native_champion_model = mlflow.sklearn.load_model(MODEL_URI)

churn_probabilities = (
    native_champion_model.predict_proba(X_sample)[:, 1]
)

classification_threshold = float(
    run.data.params["classification_threshold"]
)
watchlist_threshold = float(
    run.data.params["watchlist_threshold"]
)

official_predictions = (
    churn_probabilities >= classification_threshold
).astype(int)

risk_bands = np.select(
    [
        churn_probabilities < watchlist_threshold,
        churn_probabilities < classification_threshold,
    ],
    [
        "Low",
        "Medium",
    ],
    default="High",
)

validation_results = pd.DataFrame(
    {
        "customer_id": sample_data["customer_id"],
        "actual_churn_flag": sample_data["churn_flag"],
        "churn_probability": churn_probabilities,
        "risk_band": risk_bands,
        "official_prediction": official_predictions,
        "pyfunc_default_prediction": pyfunc_predictions,
        "model_version": int(champion_version.version),
        "model_alias": "champion",
    }
)

assert len(validation_results) == 10
assert validation_results["churn_probability"].between(0, 1).all()
assert validation_results["churn_probability"].notna().all()
assert set(validation_results["risk_band"]).issubset(
    {"Low", "Medium", "High"}
)
assert validation_results["model_version"].eq(3).all()

print("Champion inference validation passed.")
print(f"Threshold watchlist: {watchlist_threshold:.4f}")
print(f"Threshold oficial: {classification_threshold:.4f}")

display(validation_results)

Champion inference validation passed.
Threshold watchlist: 0.2075
Threshold oficial: 0.4762


,customer_id,actual_churn_flag,churn_probability,risk_band,official_prediction,pyfunc_default_prediction,model_version,model_alias
0,708082083,0,0.001440,Low,0,0,3,champion
1,708083283,1,0.985237,High,1,1,3,champion
2,708084558,1,0.787986,High,1,1,3,champion
3,708085458,0,0.000121,Low,0,0,3,champion
4,708086958,0,0.019772,Low,0,0,3,champion
5,708095133,0,0.040234,Low,0,0,3,champion
6,708098133,0,0.000031,Low,0,0,3,champion
7,708099183,0,0.005472,Low,0,0,3,champion
8,708100533,0,0.000096,Low,0,0,3,champion
9,708103608,0,0.007115,Low,0,0,3,champion


In [5]:
portfolio_query = """
SELECT *
FROM marts.mart_customer_churn_model
ORDER BY customer_id
"""

portfolio_data = pd.read_sql(portfolio_query, engine)

print(f"Clientes carregados: {len(portfolio_data):,}")

X_portfolio = portfolio_data[FEATURE_COLUMNS].copy()

portfolio_probabilities = (
    native_champion_model.predict_proba(X_portfolio)[:, 1]
)

portfolio_predictions = (
    portfolio_probabilities >= classification_threshold
).astype(int)

portfolio_risk_bands = np.select(
    [
        portfolio_probabilities < watchlist_threshold,
        portfolio_probabilities < classification_threshold,
    ],
    [
        "Low",
        "Medium",
    ],
    default="High",
)

scoring_timestamp = pd.Timestamp.now(tz="UTC")

portfolio_scoring = pd.DataFrame(
    {
        "customer_id": portfolio_data["customer_id"],
        "churn_probability": portfolio_probabilities,
        "risk_band": portfolio_risk_bands,
        "churn_prediction": portfolio_predictions,
        "model_name": MODEL_NAME,
        "model_version": int(champion_version.version),
        "model_alias": "champion",
        "scored_at": scoring_timestamp,
    }
)

# Validações do scoring completo
assert len(portfolio_scoring) == 10_127
assert portfolio_scoring["customer_id"].nunique() == 10_127
assert portfolio_scoring["customer_id"].notna().all()
assert portfolio_scoring["churn_probability"].notna().all()
assert portfolio_scoring["churn_probability"].between(0, 1).all()
assert portfolio_scoring["risk_band"].isin(
    ["Low", "Medium", "High"]
).all()
assert portfolio_scoring["model_version"].eq(3).all()
assert portfolio_scoring["model_alias"].eq("champion").all()

risk_summary = (
    portfolio_scoring
    .groupby("risk_band", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        average_probability=("churn_probability", "mean"),
    )
)

risk_order = pd.CategoricalDtype(
    categories=["Low", "Medium", "High"],
    ordered=True,
)

risk_summary["risk_band"] = (
    risk_summary["risk_band"].astype(risk_order)
)

risk_summary = risk_summary.sort_values("risk_band")

top_risk_customers = (
    portfolio_scoring
    .sort_values("churn_probability", ascending=False)
    .head(10)
)

print("Full portfolio scoring validation passed.")
print(f"Clientes pontuados: {len(portfolio_scoring):,}")
print(f"Versão utilizada: {champion_version.version}")
print(f"Alias utilizado: champion")
print(f"Executado em: {scoring_timestamp}")

print("\nDistribuição por faixa de risco:")
display(risk_summary)

print("\nDez clientes com maior risco:")
display(top_risk_customers)

Clientes carregados: 10,127
Full portfolio scoring validation passed.
Clientes pontuados: 10,127
Versão utilizada: 3
Alias utilizado: champion
Executado em: 2026-07-18 12:38:19.920397+00:00

Distribuição por faixa de risco:


,risk_band,customers,average_probability
1,Low,8354,0.010772
2,Medium,174,0.314557
0,High,1599,0.920452



Dez clientes com maior risco:


,customer_id,churn_probability,risk_band,churn_prediction,model_name,model_version,model_alias,scored_at
9473,809849358,0.999928,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
397,708892833,0.999923,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
5353,718524783,0.999920,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
812,709745508,0.999910,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
6290,720460083,0.999891,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
5377,718577958,0.999887,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
3408,714602658,0.999883,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
6897,752604633,0.999874,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
7781,778867533,0.999863,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00
7344,771422883,0.999850,High,1,finpulse-churn-catboost,3,champion,2026-07-18 12:38:19.920397+00:00


In [6]:
from sqlalchemy import text

PREDICTIONS_TABLE = "mart_customer_churn_predictions"
PREDICTIONS_SCHEMA = "marts"

portfolio_scoring.to_sql(
    name=PREDICTIONS_TABLE,
    schema=PREDICTIONS_SCHEMA,
    con=engine,
    if_exists="fail",
    index=False,
    chunksize=1_000,
    method="multi",
)

with engine.begin() as connection:
    connection.execute(
        text(
            """
            ALTER TABLE marts.mart_customer_churn_predictions
            ADD CONSTRAINT pk_customer_churn_predictions
                PRIMARY KEY (customer_id),
            ADD CONSTRAINT chk_churn_probability
                CHECK (
                    churn_probability >= 0
                    AND churn_probability <= 1
                ),
            ADD CONSTRAINT chk_churn_prediction
                CHECK (churn_prediction IN (0, 1)),
            ADD CONSTRAINT chk_risk_band
                CHECK (risk_band IN ('Low', 'Medium', 'High'));
            """
        )
    )

    connection.execute(
        text(
            """
            CREATE INDEX idx_churn_predictions_risk_band
            ON marts.mart_customer_churn_predictions (risk_band);
            """
        )
    )

    connection.execute(
        text(
            """
            CREATE INDEX idx_churn_predictions_probability
            ON marts.mart_customer_churn_predictions
                (churn_probability DESC);
            """
        )
    )

mart_validation = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT customer_id) AS unique_customers,
        MIN(churn_probability) AS minimum_probability,
        MAX(churn_probability) AS maximum_probability,
        COUNT(DISTINCT model_version) AS model_versions,
        MIN(model_version) AS minimum_model_version,
        MAX(model_version) AS maximum_model_version,
        MAX(scored_at) AS latest_scoring
    FROM marts.mart_customer_churn_predictions
    """,
    engine,
)

mart_risk_summary = pd.read_sql(
    """
    SELECT
        risk_band,
        COUNT(*) AS customers,
        AVG(churn_probability) AS average_probability
    FROM marts.mart_customer_churn_predictions
    GROUP BY risk_band
    ORDER BY
        CASE risk_band
            WHEN 'Low' THEN 1
            WHEN 'Medium' THEN 2
            WHEN 'High' THEN 3
        END
    """,
    engine,
)

assert mart_validation.loc[0, "total_rows"] == 10_127
assert mart_validation.loc[0, "unique_customers"] == 10_127
assert mart_validation.loc[0, "model_versions"] == 1
assert mart_validation.loc[0, "minimum_model_version"] == 3
assert mart_validation.loc[0, "maximum_model_version"] == 3

print("Prediction mart created and validated successfully.")

display(mart_validation)
display(mart_risk_summary)

Prediction mart created and validated successfully.


,total_rows,unique_customers,minimum_probability,maximum_probability,model_versions,minimum_model_version,maximum_model_version,latest_scoring
0,10127,10127,0.000003,0.999928,1,3,3,2026-07-18 12:38:19.920397+00:00


,risk_band,customers,average_probability
0,Low,8354,0.010772
1,Medium,174,0.314557
2,High,1599,0.920452


In [7]:
import hashlib
import io
import json

import boto3
import mlflow

# Identificação desta execução
SCORING_EXPERIMENT = "finpulse-churn-scoring"
SCORING_RUN_NAME = "champion-portfolio-scoring"
CURATED_BUCKET = "curated"

scoring_batch = scoring_timestamp.strftime("%Y%m%dT%H%M%SZ")
scoring_partition = scoring_timestamp.strftime("%Y-%m-%d")

snapshot_key = (
    "churn_predictions/"
    f"scored_date={scoring_partition}/"
    f"churn_predictions_{scoring_batch}.parquet"
)

snapshot_uri = f"s3://{CURATED_BUCKET}/{snapshot_key}"

# Gera o arquivo Parquet em memória
snapshot_buffer = io.BytesIO()

portfolio_scoring.to_parquet(
    snapshot_buffer,
    index=False,
)

snapshot_bytes = snapshot_buffer.getvalue()

snapshot_sha256 = hashlib.sha256(
    snapshot_bytes
).hexdigest()

# Envia o snapshot para o MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
)

s3_client.put_object(
    Bucket=CURATED_BUCKET,
    Key=snapshot_key,
    Body=snapshot_bytes,
    ContentType="application/octet-stream",
    Metadata={
        "model-name": MODEL_NAME,
        "model-version": str(champion_version.version),
        "model-alias": "champion",
        "scored-at": scoring_timestamp.isoformat(),
        "sha256": snapshot_sha256,
    },
)

# Métricas operacionais
risk_counts = (
    portfolio_scoring["risk_band"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

scoring_summary = {
    "project": "FinPulse AI",
    "source_table": "marts.mart_customer_churn_model",
    "target_table": "marts.mart_customer_churn_predictions",
    "snapshot_uri": snapshot_uri,
    "snapshot_sha256": snapshot_sha256,
    "scored_at": scoring_timestamp.isoformat(),
    "customers_scored": int(len(portfolio_scoring)),
    "unique_customers": int(
        portfolio_scoring["customer_id"].nunique()
    ),
    "model_name": MODEL_NAME,
    "model_version": int(champion_version.version),
    "model_alias": "champion",
    "watchlist_threshold": watchlist_threshold,
    "classification_threshold": classification_threshold,
    "risk_distribution": {
        "Low": int(risk_counts["Low"]),
        "Medium": int(risk_counts["Medium"]),
        "High": int(risk_counts["High"]),
    },
}

assert mlflow.active_run() is None

mlflow.set_experiment(SCORING_EXPERIMENT)

with mlflow.start_run(run_name=SCORING_RUN_NAME) as scoring_run:
    scoring_run_id = scoring_run.info.run_id

    mlflow.set_tags(
        {
            "project": "FinPulse AI",
            "pipeline": "batch_scoring",
            "problem": "bank_customer_churn",
            "model_name": MODEL_NAME,
            "model_alias": "champion",
            "source_table": (
                "marts.mart_customer_churn_model"
            ),
            "target_table": (
                "marts.mart_customer_churn_predictions"
            ),
            "snapshot_uri": snapshot_uri,
            "status": "validated",
        }
    )

    mlflow.log_params(
        {
            "model_version": int(champion_version.version),
            "watchlist_threshold": watchlist_threshold,
            "classification_threshold": (
                classification_threshold
            ),
            "feature_count": len(FEATURE_COLUMNS),
        }
    )

    mlflow.log_metrics(
        {
            "customers_scored": len(portfolio_scoring),
            "unique_customers": (
                portfolio_scoring["customer_id"].nunique()
            ),
            "low_risk_customers": int(risk_counts["Low"]),
            "medium_risk_customers": int(
                risk_counts["Medium"]
            ),
            "high_risk_customers": int(risk_counts["High"]),
            "average_churn_probability": float(
                portfolio_scoring[
                    "churn_probability"
                ].mean()
            ),
            "maximum_churn_probability": float(
                portfolio_scoring[
                    "churn_probability"
                ].max()
            ),
        }
    )

    mlflow.log_dict(
        scoring_summary,
        "scoring/scoring_summary.json",
    )

# Vincula o mart ao run e ao snapshot
with engine.begin() as connection:
    connection.execute(
        text(
            """
            ALTER TABLE
                marts.mart_customer_churn_predictions
            ADD COLUMN IF NOT EXISTS scoring_run_id TEXT,
            ADD COLUMN IF NOT EXISTS snapshot_uri TEXT;
            """
        )
    )

    connection.execute(
        text(
            """
            UPDATE marts.mart_customer_churn_predictions
            SET
                scoring_run_id = :scoring_run_id,
                snapshot_uri = :snapshot_uri;
            """
        ),
        {
            "scoring_run_id": scoring_run_id,
            "snapshot_uri": snapshot_uri,
        },
    )

print("End-to-end scoring registration passed.")
print(f"MLflow Run ID: {scoring_run_id}")
print(f"MinIO snapshot: {snapshot_uri}")
print(f"SHA-256: {snapshot_sha256}")

2026/07/18 12:43:14 INFO mlflow.tracking.fluent: Experiment with name 'finpulse-churn-scoring' does not exist. Creating a new experiment.


🏃 View run champion-portfolio-scoring at: http://mlflow:5000/#/experiments/5/runs/70766c70fc5f45d6ba2c9b4022b183e9
🧪 View experiment at: http://mlflow:5000/#/experiments/5
End-to-end scoring registration passed.
MLflow Run ID: 70766c70fc5f45d6ba2c9b4022b183e9
MinIO snapshot: s3://curated/churn_predictions/scored_date=2026-07-18/churn_predictions_20260718T123819Z.parquet
SHA-256: df9f8cc39b244a742a5c0a88bd2b277661152019a3d03ff052e0d1929d5c3754


In [8]:
# 1. Recupera novamente o snapshot do MinIO
snapshot_response = s3_client.get_object(
    Bucket=CURATED_BUCKET,
    Key=snapshot_key,
)

recovered_snapshot_bytes = (
    snapshot_response["Body"].read()
)

recovered_sha256 = hashlib.sha256(
    recovered_snapshot_bytes
).hexdigest()

recovered_scoring = pd.read_parquet(
    io.BytesIO(recovered_snapshot_bytes)
)

# 2. Recupera novamente o run do MLflow
registered_scoring_run = client.get_run(
    scoring_run_id
)

scoring_artifacts = client.list_artifacts(
    scoring_run_id,
    path="scoring",
)

# 3. Confere a linhagem gravada no mart
mart_lineage_validation = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT scoring_run_id) AS scoring_runs,
        MIN(scoring_run_id) AS scoring_run_id,
        COUNT(DISTINCT snapshot_uri) AS snapshots,
        MIN(snapshot_uri) AS snapshot_uri
    FROM marts.mart_customer_churn_predictions
    """,
    engine,
)

# 4. Validações finais
assert recovered_sha256 == snapshot_sha256
assert len(recovered_scoring) == 10_127
assert recovered_scoring["customer_id"].nunique() == 10_127
assert recovered_scoring["model_version"].eq(3).all()
assert recovered_scoring["model_alias"].eq("champion").all()

assert registered_scoring_run.info.status == "FINISHED"
assert (
    registered_scoring_run.data.metrics["customers_scored"]
    == 10_127
)

assert mart_lineage_validation.loc[0, "total_rows"] == 10_127
assert mart_lineage_validation.loc[0, "scoring_runs"] == 1
assert (
    mart_lineage_validation.loc[0, "scoring_run_id"]
    == scoring_run_id
)
assert mart_lineage_validation.loc[0, "snapshots"] == 1
assert (
    mart_lineage_validation.loc[0, "snapshot_uri"]
    == snapshot_uri
)

print("Cross-system scoring validation passed.")
print(f"MLflow status: {registered_scoring_run.info.status}")
print(f"MinIO rows recovered: {len(recovered_scoring):,}")
print(f"Checksum confirmed: {recovered_sha256}")
print(
    "MLflow artifacts:",
    [artifact.path for artifact in scoring_artifacts],
)

display(mart_lineage_validation)

Cross-system scoring validation passed.
MLflow status: FINISHED
MinIO rows recovered: 10,127
Checksum confirmed: df9f8cc39b244a742a5c0a88bd2b277661152019a3d03ff052e0d1929d5c3754
MLflow artifacts: ['scoring/scoring_summary.json']


,total_rows,unique_customers,scoring_runs,scoring_run_id,snapshots,snapshot_uri
0,10127,10127,1,70766c70fc5f45d6ba2c9b4022b183e9,1,s3://curated/churn_predictions/scored_date=202...
